In [1]:
using PowerModelsITD
import PowerModelsONM as _ONM
import PowerModelsDistribution as _PMD
import PowerModels as _PM
import InfrastructureModels as _IM

In [2]:
using JSON
using UUIDs

In [26]:
#using JuMP, Ipopt, Juniper
#optimizer = Juniper.Optimizer
#nl_solver = optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0)

import Gurobi
solver = _ONM.optimizer_with_attributes(() -> Gurobi.Optimizer(_ONM.GRB_ENV), "Presolve" => 0, "FeasibilityTol" => 1E-4)
#solver = _ONM.optimizer_with_attributes(() -> Juniper.Optimizer(), "nl_solver" => nl_solver)#"NumericFocus" => 3, "BarHomogeneous" => 1, "BarQCPConvTol" => 1E-1, "DualReductions" => 0)

MathOptInterface.OptimizerWithAttributes(var"#23#24"(), Pair{MathOptInterface.AbstractOptimizerAttribute, Any}[MathOptInterface.RawOptimizerAttribute("Presolve") => 0, MathOptInterface.RawOptimizerAttribute("FeasibilityTol") => 0.0001])

In [27]:
pmitd_type = BFPowerModelITD{BFAPowerModel, _PMD.LinDist3FlowPowerModel} 

BFPowerModelITD{BFAPowerModel, LPUBFDiagPowerModel} (alias for AbstractBFPowerModelITD{BFAPowerModel, LPUBFDiagPowerModel})

In [28]:
using JLD2
itd_data = JLD2.load("/Users/rferrando/Desktop/CATS_SFO_merged_ITD.jld2")["full_dict"]

┌ JLD2 | Warning ] : type Main.PowerModelsDistribution.ConnConfig does not exist in workspace; reconstructing
└ @ JLD2 ~/.julia/packages/JLD2/hbsZG/src/data/reconstructing_datatypes.jl:472
┌ JLD2 | Warning ] : type Main.PowerModelsDistribution.LoadModel does not exist in workspace; reconstructing
└ @ JLD2 ~/.julia/packages/JLD2/hbsZG/src/data/reconstructing_datatypes.jl:472
┌ JLD2 | Warning ] : type Main.PowerModelsDistribution.DataModel does not exist in workspace; reconstructing
└ @ JLD2 ~/.julia/packages/JLD2/hbsZG/src/data/reconstructing_datatypes.jl:472


Dict{String, Any} with 3 entries:
  "multiinfrastructure" => true
  "it"                  => Dict{Any, Any}("pmd"=>Dict{String, Any}("name"=>"fee…
  "per_unit"            => true

In [29]:
itd_data["it"]["pmd"]["nw"]["3"]["load"]["100"]

Dict{String, Any} with 13 entries:
  "source_id"     => "EnergyConsumer.load_p31ulv8555"
  "load_bus"      => 130
  "model"         => Reconstruct@LoadModel(0)
  "connections"   => [1, 2, 3]
  "dispatchable"  => 0
  "index"         => 100
  "configuration" => Reconstruct@ConnConfig(0)
  "name"          => "feeder_p31udt2982-p31uhs18_1247x.load_p31ulv8555"
  "status"        => 1
  "qd"            => [4.164e-8, 4.164e-8, 4.164e-8]
  "vbase"         => 0.277128
  "vnom_kv"       => 1.0
  "pd"            => [1.51284e-7, 1.51284e-7, 1.51284e-7]

In [30]:
feeder_name = itd_data["it"]["pmd"]["name"]


nw_data = deepcopy(itd_data["it"]["pmd"])
bus_keys = collect(keys(nw_data["nw"]["1"]["bus"]))

nw_keys = ["1", "2", "3"]
pm_orig = deepcopy(itd_data["it"]["pm"])
pmitd_orig = deepcopy(itd_data["it"]["pmitd"])
itd_data["it"]["pm"]["nw"] = Dict("1" => pm_orig, "2" => pm_orig, "3" => pm_orig)
itd_data["it"]["pmitd"]["nw"] = Dict("1" => pmitd_orig, "2" => pmitd_orig, "3" => pmitd_orig)
itd_data["it"]["pmitd"]["multinetwork"] = true
itd_data["it"]["pm"]["multinetwork"] = true
pop!(itd_data["it"]["pmitd"], "100001")

Dict{String, Any} with 2 entries:
  "distribution_boundary" => "feeder_p31udt2982-p31uhs18_1247x.voltage_source.s…
  "transmission_boundary" => "7957"

In [31]:
itd_data["it"]["pmd"] = deepcopy(nw_data)

Dict{String, Any} with 7 entries:
  "name"                    => "feeder_p31udt2982-p31uhs18_1247x"
  "map"                     => Dict{String, Vector{Dict{String, Any}}}("1"=>[Di…
  "bus_lookup"              => Dict{String, Dict{String, Any}}("1"=>Dict("p31ud…
  "switch_close_actions_ub" => Inf
  "multinetwork"            => true
  "nw"                      => Dict{String, Any}("1"=>Dict{String, Any}("is_kro…
  "data_model"              => Reconstruct@DataModel(1)

In [32]:
itd_data["it"]["pmd"] = deepcopy(nw_data)

Dict{String, Any} with 7 entries:
  "name"                    => "feeder_p31udt2982-p31uhs18_1247x"
  "map"                     => Dict{String, Vector{Dict{String, Any}}}("1"=>[Di…
  "bus_lookup"              => Dict{String, Dict{String, Any}}("1"=>Dict("p31ud…
  "switch_close_actions_ub" => Inf
  "multinetwork"            => true
  "nw"                      => Dict{String, Any}("1"=>Dict{String, Any}("is_kro…
  "data_model"              => Reconstruct@DataModel(1)

In [33]:
nw_keys = ["1", "2", "3"]
load_keys = collect(keys(itd_data["it"]["pmd"]["nw"]["1"]["load"]))
for nw in nw_keys
    for load in load_keys
        itd_data["it"]["pmd"]["nw"][nw]["load"][load]["model"] = _PMD.POWER
        itd_data["it"]["pmd"]["nw"][nw]["load"][load]["configuration"] = _PMD.WYE
    end
end


In [34]:
itd_data["it"]["pm"]["nw"]["1"]["load"]["32"]

Dict{String, Any} with 4 entries:
  "load_bus" => 3107
  "status"   => 1
  "qd"       => 0.0
  "pd"       => 0.0

In [35]:
nw_keys = ["1", "2", "3"]
load_keys_pm = collect(keys(itd_data["it"]["pm"]["nw"]["1"]["load"]))
for nw in nw_keys
    for load in load_keys_pm
        itd_data["it"]["pm"]["nw"][nw]["load"][load]["index"] = parse(Int, load)
    end
end


In [36]:
load_keys = collect(keys(itd_data["it"]["pmd"]["nw"]["1"]["load"]))
#storage_keys = collect(keys(itd_data["it"]["pmd"]["nw"]["1"]["storage"]))

net_load_p = 0
for nw in nw_keys
    for load in load_keys
        itd_data["it"]["pmd"]["nw"][nw]["load"][load]["pd"] = Float64.(itd_data["it"]["pmd"]["nw"][nw]["load"][load]["pd"])
        itd_data["it"]["pmd"]["nw"][nw]["load"][load]["qd"] = Float64.(itd_data["it"]["pmd"]["nw"][nw]["load"][load]["qd"])
        net_load_p += sum(Float64.(itd_data["it"]["pmd"]["nw"][nw]["load"][load]["pd"]))
    end
end

#=
for nw in nw_keys
    for storage in storage_keys
        itd_data["it"]["pmd"]["nw"][nw]["storage"][storage]["connections"] = Int64.(itd_data["it"]["pmd"]["nw"][nw]["storage"][storage]["connections"])
    end
end
=#

In [37]:
#nw_d = _PMD.parse_file("/Users/rferrando/Desktop/feeder_p31udt2982_v2/feeder_outputs/Master.dss", multinetwork=true)

In [38]:
#itd_data["it"]["pmd"] = deepcopy(nw_d_math)

In [39]:
itd_data["it"]["pmd"]["nw"]["1"]["bus"]["407"]["vmin"]

3-element Vector{Float64}:
 0.0
 0.0
 0.0

In [40]:
bus_keys = collect(keys(itd_data["it"]["pmd"]["nw"]["1"]["bus"]))
#storage_keys = collect(keys(itd_data["it"]["pmd"]["nw"]["1"]["storage"]))


for nw in nw_keys
    for bus in bus_keys
        itd_data["it"]["pmd"]["nw"][nw]["bus"][bus]["vmin"] = [0.92, 0.92, 0.92]
        itd_data["it"]["pmd"]["nw"][nw]["bus"][bus]["vmax"] = [1.1, 1.1, 1.1]
    end
end


In [41]:
itd_data["it"]["pm"]["nw"]["1"]["load"]["1"]

Dict{String, Any} with 5 entries:
  "load_bus" => 4
  "status"   => 1
  "qd"       => 8.37325
  "pd"       => 26.0243
  "index"    => 1

In [42]:
#=
nw_keys = collect(keys(itd_data["it"]["pmd"]["nw"]))
for nw in nw_keys
    itd_data["it"]["pmd"]["nw"][nw]["switch_close_actions_ub"] = Inf
    #open all switches, should close successively
    switch_keys = collect(keys(itd_data["it"]["pmd"]["nw"][nw]["switch"]))
    for sw in switch_keys
        if nw == "1"
            itd_data["it"]["pmd"]["nw"][nw]["switch"][sw]["state"] = Int(_ONM.OPEN)
            itd_data["it"]["pmd"]["nw"][nw]["switch"][sw]["dispatchable"] = Int(_ONM.NO)
        else
            itd_data["it"]["pmd"]["nw"][nw]["switch"][sw]["state"] = Int(_ONM.OPEN)
            itd_data["it"]["pmd"]["nw"][nw]["switch"][sw]["dispatchable"] = Int(_ONM.YES)
        end
    end
end
=#


In [43]:
_PM.make_per_unit!(itd_data["it"]["pm"]["nw"]["1"])

In [44]:
_PM.make_per_unit!(itd_data["it"]["pm"]["nw"]["2"])

In [45]:
_PM.make_per_unit!(itd_data["it"]["pm"]["nw"]["3"])

In [46]:
itd_data["it"]["pmitd"]["nw"]["3"]["100018"]

Dict{String, Any} with 2 entries:
  "distribution_boundary" => "feeder_p31udt2982-p31uhs18_1247x.voltage_source.s…
  "transmission_boundary" => "7824"

In [47]:
using JuMP
using Ipopt
include("../src/prob/opfitd_restore.jl")
include("../src/prob/ONM_objective_mod.jl")
r = solve_mn_opfitd(itd_data, pmitd_type, Ipopt.Optimizer, solution_model="MATH")
# dist_bus_name -- check refs for passing dict instead of copying it

[ PowerModelsDistribution | Info ] : Data model 'Reconstruct@DataModel(1)' is not recognized, no model type transformation performed


dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_bus_name = voltage_source.feeder_p31udt2982-p31uhs18_1247x.source
dist_b

[ PowerModelsITD | Info ] : pmitd model solution time (instantiate + optimization): 31.493921995162964


Dict{String, Any} with 8 entries:
  "solve_time"         => 30.6797
  "optimizer"          => "Ipopt"
  "termination_status" => LOCALLY_INFEASIBLE
  "dual_status"        => INFEASIBLE_POINT
  "primal_status"      => INFEASIBLE_POINT
  "objective"          => 2.58722e5
  "solution"           => Dict{String, Any}("multiinfrastructure"=>true, "it"=>…
  "objective_lb"       => -Inf

In [48]:
itd_data["it"]["pmd"]["nw"]["3"]["branch"]["101"]

Dict{String, Any} with 23 entries:
  "br_r"          => [0.00107509 0.000193598 0.000193598; 0.000193598 0.0010750…
  "rate_a"        => [Inf, Inf, Inf]
  "f_connections" => [1, 2, 3]
  "rate_b"        => [Inf, Inf, Inf]
  "name"          => "l(r:p31udt6096-p31udt6099)"
  "br_x"          => [0.0029664 0.0021725 0.0021725; 0.0021725 0.0029664 0.0021…
  "rate_c"        => [Inf, Inf, Inf]
  "g_to"          => [0.0 0.0 0.0; 0.0 0.0 0.0; 0.0 0.0 0.0]
  "c_rating_a"    => [4.31973, 4.31973, 4.31973]
  "vbase"         => 7.19956
  "g_fr"          => [0.0 0.0 0.0; 0.0 0.0 0.0; 0.0 0.0 0.0]
  "source_id"     => "ACLineSegment.l(r:p31udt6096-p31udt6099)"
  "t_connections" => [1, 2, 3]
  "f_bus"         => 253
  "b_fr"          => [0.000199106 -9.77019e-5 -9.77019e-5; -9.77019e-5 0.000199…
  "c_rating_b"    => [4.31973, 4.31973, 4.31973]
  "br_status"     => 1
  "t_bus"         => 200
  "b_to"          => [0.000199106 -9.77019e-5 -9.77019e-5; -9.77019e-5 0.000199…
  "index"         => 101
  "angm

In [50]:
r["solution"]["it"]["pmd"]["nw"]["3"]["branch"]["101"]

Dict{String, Any} with 4 entries:
  "qf" => [10.0747, 10.0747, 10.0748]
  "qt" => [-10.5107, -10.5107, -10.5107]
  "pt" => [-2600.04, -2600.05, -2600.05]
  "pf" => [2600.04, 2600.05, 2600.05]

In [ ]:
ws = []
for nw in nw_keys
    for bus in bus_keys
        try 
            push!(ws, r["solution"]["it"]["pmd"]["nw"][nw]["bus"][string(bus)]["w"][1], 
            r["solution"]["it"]["pmd"]["nw"][nw]["bus"][string(bus)]["w"][2], 
            r["solution"]["it"]["pmd"]["nw"][nw]["bus"][string(bus)]["w"][3])
        catch
            2 > 1
        end
    end
end
            

In [ ]:
sqrt(maximum(ws))

In [ ]:
r["solution"]["it"]["pmd"]["nw"]["3"]["branch"]["12"]

In [ ]:
itd_data["it"]["pmd"]["nw"]["3"]["gen"]["1"]

In [ ]:
itd_data["it"]["pmd"]["nw"]["2"]["bus"]["14"]

In [ ]:
r["solution"]["it"]["pmd"]["nw"]["2"]["switch"]["1"]

In [ ]:
itd_data["it"]["pm"]["nw"]["1"]["bus"]["1"]

In [ ]:
function create_transmission_settings!(itd_data, r)
    nw_keys = collect(keys(itd_data["it"]["pm"]["nw"]))
    for nw in nw_keys
        itd_data["it"]["pm"]["nw"][nw]["settings"] = deepcopy(itd_data["it"]["pmd"]["nw"][nw]["settings"])
        itd_data["it"]["pm"]["nw"][nw]["settings"]["sbase_default"] = itd_data["it"]["pm"]["nw"][nw]["baseMVA"]
        itd_data["it"]["pm"]["nw"][nw]["settings"]["vbases_default"] = Dict{String, Real}()
        for bus in collect(keys(r["solution"]["it"]["pm"]["nw"][nw]["bus"]))
            push!(itd_data["it"]["pm"]["nw"][nw]["settings"]["vbases_default"], bus => itd_data["it"]["pm"]["nw"][nw]["bus"][bus]["base_kv"])
        end
        itd_data["it"]["pm"]["nw"][nw]["settings"]["voltage_scale_factor"] = 1000.0 # assuming everything is in kV
        itd_data["it"]["pm"]["nw"][nw]["settings"]["sbase"] = itd_data["it"]["pm"]["nw"][nw]["baseMVA"]
        itd_data["it"]["pm"]["nw"][nw]["settings"]["power_scale_factor"] = itd_data["it"]["pm"]["nw"][nw]["baseMVA"]
        r["solution"]["it"]["pm"]["nw"][nw]["settings"] = deepcopy(itd_data["it"]["pm"]["nw"][nw]["settings"])
    end
end

In [ ]:
create_transmission_settings!(itd_data, r)

In [ ]:
itd_data["it"]["pm"]["nw"]["1"]["settings"]

In [ ]:
include("/Users/rferrando/Documents/CUI-code/PowerModelsDistribution-jl/src/PowerModelsDistribution.jl")
import .PowerModelsDistribution as __PMD


In [ ]:
function extract_first_number(str)
    # Remove the parentheses
    str = strip(str, ['(', ')'])

    # Split the string by the comma
    nums = split(str, ",")

    # Convert the first element to an integer
    num1 = parse(Int, nums[1])

    return num1
end


include("/Users/rferrando/Documents/CUI-code/PowerModelsDistribution-jl/src/data_model/units.jl")
include("/Users/rferrando/Documents/CUI-code/PowerModelsDistribution-jl/src/data_model/transformations/math2ravens-transmission.jl")

function transform_solution_ravens_itd(res, full_nw_math, ravens_t_file)
    sol_t_r = transform_solution_ravens_transmission(r["solution"]["it"]["pm"], itd_data["it"]["pm"])["AnalysisResult"]

    res_d = res["solution"]["it"]["pmd"]
    sol_d_rv = PowerModelsDistribution.transform_solution_ravens(res_d, full_nw_math["it"]["pmd"])["AnalysisResult"]

    bd_keys = collect(keys(res["solution"]["it"]["pmitd"]["nw"]["1"]["boundary"]))
    println(bd_keys)

    # record boundary flows -- this part is very messy and relies on the specific way I named the boundary. Will need to be revisited.
    nw_keys_math = keys(res["solution"]["it"]["pmitd"]["nw"])
    phases = ["SinglePhaseKind.A", "SinglePhaseKind.B", "SinglePhaseKind.C"]
    println(nw_keys_math)
    bd_count = 0
    for k in 1:Int(length(bd_keys)/2)
        bd_count += 1
        bd_ind = extract_first_number(bd_keys[bd_count]) - 100000
        from_ind = bd_keys[2*bd_count]
        to_ind = bd_keys[2*bd_count-1]
        for phase in 1:length(phases)
        data1 = Dict(
                "AnalysisResultData.Curve" => 
                __PMD._make_curve_data(nw_keys_math, full_nw_math["it"]["pmitd"], nw -> 
                    begin
                    Dict("AvPowerFlow.p" => res["solution"]["it"]["pmitd"]["nw"][nw]["boundary"][from_ind]["pbound_fr"][1]*res["solution"]["it"]["pm"]["nw"][nw]["settings"]["power_scale_factor"],
                                    "AvPowerFlow.q" => res["solution"]["it"]["pmitd"]["nw"][nw]["boundary"][from_ind]["qbound_fr"][1]*res["solution"]["it"]["pm"]["nw"][nw]["settings"]["power_scale_factor"])
        
                    end
                ),
                 "ArPowerFlow.ConductingEquipment" => "ACLineSegment::'line_bd_"*string(bd_ind)*"'",
                 "AnalysisResultData.phase"        => phases[phase]
                
        )
        data2 = Dict(
                "AnalysisResultData.Curve" => 
                __PMD._make_curve_data(nw_keys_math, full_nw_math["it"]["pmitd"], nw -> 
                    begin
                    Dict("AvPowerFlow.p" => res["solution"]["it"]["pmitd"]["nw"][nw]["boundary"][to_ind]["pbound_to"][phase]*res["solution"]["it"]["pmd"]["nw"][nw]["settings"]["power_scale_factor"],
                                    "AvPowerFlow.q" => res["solution"]["it"]["pmitd"]["nw"][nw]["boundary"][to_ind]["qbound_to"][phase]*res["solution"]["it"]["pmd"]["nw"][nw]["settings"]["power_scale_factor"], "AnalysisResultData.Phase" => phases[phase])
                    end
                ),
                 "ArPowerFlow.ConductingEquipment" => "ACLineSegment::'line_bd_"*string(bd_ind)*"'",
                 "AnalysisResultData.phase"        => phases[phase]
                
        )
        push!(sol_t_r["OptimalPowerFlow"]["OperationsResult.PowerFlows"], data1, data2)
        end
    end

    # add everything to the transmission solution
    for k in collect(keys(sol_d_rv["OptimalPowerFlow"]["OperationsResult.Voltages"]))
        push!(sol_t_r["OptimalPowerFlow"]["OperationsResult.Voltages"], sol_d_rv["OptimalPowerFlow"]["OperationsResult.Voltages"][k])
    end
    for k in collect(keys(sol_d_rv["OptimalPowerFlow"]["OperationsResult.PowerFlows"]))
        push!(sol_t_r["OptimalPowerFlow"]["OperationsResult.PowerFlows"], sol_d_rv["OptimalPowerFlow"]["OperationsResult.PowerFlows"][k])
    end    
    sol_t_r["OptimalPowerFlow"]["OperationsResult.Statuses"] = deepcopy(sol_d_rv["OptimalPowerFlow"]["OperationsResult.Statuses"])
    sol_t_r["OptimalPowerFlow"]["OperationsResult.Switches"] = deepcopy(sol_d_rv["OptimalPowerFlow"]["OperationsResult.Switches"])

    return sol_t_r


end


In [ ]:
ravens_file = "/Users/rferrando/Desktop/case5_case3_merged_bal.json"

In [ ]:
itd_data["it"]["pm"]["data_model"] = __PMD.MATHEMATICAL
itd_data["it"]["pmd"]["data_model"] = __PMD.MATHEMATICAL
itd_data["it"]["pm"]["per_unit"] = true
r["solution"]["it"]["pm"]["nw"]["1"]["per_unit"] = true
r["solution"]["it"]["pm"]["nw"]["2"]["per_unit"] = true
itd_data["it"]["pm"]["nw"]["1"]["time_elapsed"] = 0.0
itd_data["it"]["pm"]["nw"]["2"]["time_elapsed"] = 1.0
itd_data["it"]["pmitd"]["nw"]["1"]["time_elapsed"] = 0.0
itd_data["it"]["pmitd"]["nw"]["2"]["time_elapsed"] = 1.0


In [ ]:
sol_ravens = transform_solution_ravens_itd(r, itd_data, ravens_file)

In [ ]:
###### sol_ravens["OptimalPowerFlow"]["OperationsResult.Switches"][1]["AnalysisResultData.Curve"]["AnalysisResultCurve.CurveDatas"]

In [ ]:
sol_ravens["OptimalPowerFlow"]["OperationsResult.PowerFlows"][13]["AnalysisResultData.Curve"]["AnalysisResultCurve.CurveDatas"]

In [ ]:
using JSON
open("/Users/rferrando/Desktop/case5_case3_results_ravens.json","w") do f
  JSON.print(f, sol_ravens, 2)
end

In [ ]:
nw_r = JSON.parsefile(ravens_file)

In [ ]:
sol_ravens_dict = Dict("AnalysisResult" => sol_ravens)

In [ ]:
full_dict = merge(nw_r, sol_ravens_dict)

In [ ]:
using JSON
open("/Users/rferrando/Desktop/case5_case3_nw_and_results_ravens.json","w") do f
  JSON.print(f,full_dict, 2)
end

In [ ]:
include("/Users/rferrando/Documents/CUI-code/PowerModelsDistribution-jl/src/PowerModelsDistribution.jl")
import .PowerModelsDistribution as __PMD


In [ ]:
itd_data["it"]["pmd"]["data_model"] = __PMD.MATHEMATICAL

In [ ]:
sol_d_rv = __PMD.transform_solution_ravens(r["solution"]["it"]["pmd"], itd_data["it"]["pmd"])

In [ ]:
itd_data["it"]["pm"]["data_model"] = __PMD.MATHEMATICAL
itd_data["it"]["pm"]["per_unit"] = true


In [ ]:
r["solution"]["it"]["pm"]["nw"]["1"]["per_unit"] = true

In [ ]:
r["solution"]["it"]["pm"]["nw"]["2"]["per_unit"] = true

In [ ]:
itd_data["it"]["pm"]["nw"]["1"]["time_elapsed"] = 0.0

In [ ]:
itd_data["it"]["pm"]["nw"]["2"]["time_elapsed"] = 1.0


In [ ]:
r["solution"]["it"]["pm"]["nw"]["1"]

In [ ]:
include("/Users/rferrando/Documents/CUI-code/PowerModelsDistribution-jl/src/data_model/transformations/math2ravens-transmission.jl")
sol_t_rv = transform_solution_ravens_transmission(r["solution"]["it"]["pm"], itd_data["it"]["pm"])

In [ ]:
sol_t_rv["AnalysisResult"]["OptimalPowerFlow"]

In [ ]:
r["solution"]["it"]["pmitd"]["nw"]["2"]["boundary"]["(100001, 5, 5)"]

In [ ]:
sol_d_rv["AnalysisResult"]["OptimalPowerFlow"]["OperationsResult.PowerFlows"]

In [ ]:
push!(sol_t_rv["AnalysisResult"]["OptimalPowerFlow"]["OperationsResult.Voltages"], sol_d_rv["AnalysisResult"]["OptimalPowerFlow"]["OperationsResult.Voltages"])
push!(sol_t_rv["AnalysisResult"]["OptimalPowerFlow"]["OperationsResult.PowerFlows"], sol_d_rv["AnalysisResult"]["OptimalPowerFlow"]["OperationsResult.PowerFlows"])
sol_t_rv["AnalysisResult"]["OptimalPowerFlow"]["OperationsResult.Statuses"] = deepcopy(sol_d_rv["AnalysisResult"]["OptimalPowerFlow"]["OperationsResult.Statuses"])
sol_t_rv["AnalysisResult"]["OptimalPowerFlow"]["OperationsResult.Switches"] = deepcopy(sol_d_rv["AnalysisResult"]["OptimalPowerFlow"]["OperationsResult.Switches"])

In [ ]:
function sol_data_model_itdonm!(pm::AbstractBFPowerModelITD, solution::Dict{String,<:Any})
    solution_d = solution["it"]["pmd"]
    _PMD.apply_pmd!(_PMD._sol_data_model_w!, solution_d)
end

function solution_statuses_itdonm!(pm::AbstractBFPowerModelITD, sol::Dict{String,Any})
    solution_d = sol["it"]["pmd"]
    _PMD.apply_pmd!(_ONM._solution_statuses!, solution_d; apply_to_subnetworks=true)
end


In [ ]:
mod = instantiate_model(itd_data, pmitd_type, build_mn_opfitd_restore, multinetwork=true, 
    pmitd_ref_extensions = vcat(Function[], _ONM._default_ref_extensions...))

In [ ]:
using JuMP
JuMP.set_optimizer(mod.model, solver)

In [ ]:
JuMP.optimize!(mod.model)

In [ ]:
gurobi_model = backend(mod.model).optimizer.model

In [ ]:
MOI.compute_conflict!(gurobi_model)

In [ ]:
println("\n=== CONFLICTING CONSTRAINTS ===\n")
conflict_count = 0

for (F, S) in MOI.get(gurobi_model, MOI.ListOfConstraintTypesPresent())
    cons = MOI.get(gurobi_model, MOI.ListOfConstraintIndices{F,S}())
    for con in cons
        try
            status = MOI.get(gurobi_model, MOI.ConstraintConflictStatus(), con)
            if status == MOI.IN_CONFLICT
                conflict_count += 1
                println("[$conflict_count] Constraint $con:")

                try
                    con_func = MOI.get(gurobi_model, MOI.ConstraintFunction(), con)
                    con_set = MOI.get(gurobi_model, MOI.ConstraintSet(), con)

                    print("    ")
                    if con_set isa MOI.LessThan
                        println("$(con_func) ≤ $(con_set.upper)")
                    elseif con_set isa MOI.GreaterThan
                        println("$(con_func) ≥ $(con_set.lower)")
                    elseif con_set isa MOI.EqualTo
                        println("$(con_func) = $(con_set.value)")
                    else
                        println("$(con_func) ∈ $(con_set)")
                    end
                catch
                    println("    (Unable to retrieve details)")
                end
                println()
            end
        catch
            continue
        end
    end
end

println("Total: $conflict_count constraints in conflict")


In [ ]:
r["solution"]["it"]["pmd"]["nw"]["2"]["bus"]["1"]

In [ ]:
#THE PROBLEM IS WITH SM_UB